# Smart Laptop Price Predictor - Training Notebook (Google Colab)

This notebook trains and compares 8 regression models (plus a stacking ensemble) to predict laptop prices from specifications, then exports a single `best_model.pkl` artifact that the FastAPI backend loads.

**Pipeline:** Data -> Preprocessing -> Train 8 Models + Tuning -> Stacking Ensemble -> Evaluate -> Auto-select Best -> Save Artifact.

**Improvements over basic:**
- RandomizedSearchCV hyperparameter tuning for tree-based models
- Stacking ensemble combining the top 4 models
- Wider hyperparameter search spaces


## 1. Install dependencies

In [ ]:
%%capture
!pip install -q pandas numpy scikit-learn xgboost catboost lightgbm joblib matplotlib seaborn


## 2. (Optional) Use a REAL Kaggle dataset

1. Create a Kaggle account and download `kaggle.json` from Account -> API.
2. Upload it in the file pane on the left, then run the next cell.
3. Set `DATASET_SLUG` and `DATA_FILE` to your chosen dataset.

In [ ]:
import os, json, zipfile

if os.path.exists('kaggle.json'):
    os.makedirs('/root/.kaggle', exist_ok=True)
    os.system('cp kaggle.json /root/.kaggle/kaggle.json')
    os.chmod('/root/.kaggle/kaggle.json', 600)
    print('Kaggle credentials configured.')
else:
    print('kaggle.json not found -> will use synthetic data.')

DATASET_SLUG = 'bhavikpatel9473/laptop-price-prediction'
DATA_FILE = 'laptop_price.csv'
RAW_CSV = None
if os.path.exists('/root/.kaggle/kaggle.json'):
    !kaggle datasets download -d {DATASET_SLUG} --unzip -o ./kaggle_data
    for root, _, files in os.walk('./kaggle_data'):
        if DATA_FILE in files:
            RAW_CSV = os.path.join(root, DATA_FILE)
    print('Raw dataset:', RAW_CSV)


## 3. Normalise a Kaggle dataset into the project schema

In [ ]:
import re, numpy as np, pandas as pd

def parse_cpu(cpu):
    cpu = str(cpu)
    brand = 'Intel'
    if 'amd' in cpu.lower() or 'ryzen' in cpu.lower():
        brand = 'AMD'
    elif 'apple' in cpu.lower() or 'm1' in cpu.lower() or 'm2' in cpu.lower() \
         or 'm3' in cpu.lower() or 'm4' in cpu.lower():
        brand = 'Apple Silicon'
    series = 'i5'
    m = re.search(r'i([3579])', cpu)
    if m: series = 'i' + m.group(1)
    m = re.search(r'Ryzen\s*([3579])', cpu, re.I)
    if m: series = 'Ryzen ' + m.group(1)
    m = re.search(r'M([1234])', cpu, re.I)
    if m: series = 'M' + m.group(1)
    return brand, series

def parse_memory(mem):
    mem = str(mem).lower()
    if 'nvme' in mem: stype = 'NVMe SSD'
    elif 'ssd' in mem: stype = 'SSD'
    elif 'hdd' in mem or 'hybrid' in mem: stype = 'HDD'
    else: stype = 'SSD'
    m = re.search(r'(\d+)\s*(gb|tb)', mem)
    size = 512
    if m:
        size = int(m.group(1))
        if m.group(2) == 'tb': size *= 1024
    return stype, size

def normalise_kaggle(path):
    df = pd.read_csv(path)
    df.columns = [c.strip() for c in df.columns]
    out = pd.DataFrame()
    out['Brand'] = df.get('Company', df.get('Brand'))
    out['Display_Size'] = pd.to_numeric(df.get('Inches', df.get('Display_Size')), errors='coerce')
    pb, ps = zip(*df.get('Cpu', df.get('Processor')).map(parse_cpu)) \
        if 'Cpu' in df.columns or 'Processor' in df.columns else (['Intel']*len(df), ['i5']*len(df))
    out['Processor_Brand'] = pb
    out['Processor_Series'] = ps
    out['RAM'] = df.get('Ram', df.get('RAM')).astype(str).str.extract(r'(\d+\s*GB)')[0].str.replace(' ', '') + '' if 'Ram' in df.columns or 'RAM' in df.columns else '8GB'
    if 'Memory' in df.columns or 'Storage' in df.columns:
        st, sz = zip(*df.get('Memory', df.get('Storage')).map(parse_memory))
        out['Storage_Type'] = st
        out['Storage_Size'] = sz
    else:
        out['Storage_Type'] = 'SSD'; out['Storage_Size'] = 512
    out['GPU'] = df.get('Gpu', df.get('GPU')).astype(str).str.split().str[:2].str.join(' ')
    out['Operating_System'] = df.get('OpSys', df.get('Operating_System')).astype(str).str.extract(r'(Windows|MacOS|Linux|Chrome|No OS)')[0]
    out['Operating_System'] = out['Operating_System'].replace({'No OS': 'DOS', 'Chrome': 'Linux'})
    out['Weight'] = pd.to_numeric(df.get('Weight', df.get('Weight')).astype(str).str.extract(r'([\d.]+)')[0], errors='coerce')
    out['Price'] = pd.to_numeric(df.get('Price', df.get('Price')), errors='coerce')
    out['Resolution'] = 'FHD'
    out['Refresh_Rate'] = 60
    out['Touch_Screen'] = 'No'
    out['Battery'] = 70
    return out.dropna(subset=['Price']).reset_index(drop=True)

df = normalise_kaggle(RAW_CSV) if RAW_CSV else None
print('Kaggle rows:', None if df is None else len(df))


## 4. Synthetic fallback generator

Generates 6000 synthetic rows with realistic price-weight relationships.

In [ ]:
def generate_synthetic(n=6000, seed=42):
    rng = np.random.default_rng(seed)
    brands = ['Apple','Dell','HP','Lenovo','Asus','Acer','MSI','Samsung','Huawei','Microsoft']
    pb = rng.choice(['Intel','AMD','Apple Silicon'], n, p=[0.6,0.3,0.1])
    series = np.array([rng.choice(['i3','i5','i7','i9'] if p=='Intel' else ['Ryzen 3','Ryzen 5','Ryzen 7','Ryzen 9'] if p=='AMD' else ['M1','M2','M3','M4']) for p in pb])
    ram = rng.choice(['4GB','8GB','16GB','24GB','32GB','64GB'], n, p=[0.1,0.3,0.4,0.1,0.07,0.03])
    st = rng.choice(['HDD','SSD','NVMe SSD'], n, p=[0.1,0.6,0.3])
    sz = rng.choice([128,256,512,1024,2048], n, p=[0.1,0.3,0.4,0.15,0.05])
    gpu = rng.choice(['Integrated','Intel Iris Xe','RTX 2050','RTX 3050','RTX 4050','RTX 4060','RTX 4070','RX 7600'], n)
    disp = rng.choice([13.0,14.0,15.6,16.0,17.0], n, p=[0.15,0.3,0.35,0.1,0.1])
    res = rng.choice(['HD','FHD','2K','4K'], n, p=[0.1,0.6,0.25,0.05])
    ref = rng.choice([60,120,144,165,240], n, p=[0.5,0.2,0.18,0.08,0.04])
    os_ = rng.choice(['Windows','Linux','MacOS','DOS'], n, p=[0.8,0.08,0.08,0.04])
    touch = rng.choice(['Yes','No'], n, p=[0.25,0.75])
    batt = rng.choice([40,56,70,90], n)
    weight = np.clip(np.array([1.9]*n) * rng.normal(1.0,0.12,n), 0.8, 4.5).round(2)
    W = {'Apple':1.55,'MSI':1.40,'Dell':1.22,'HP':1.12,'Microsoft':1.18,'Lenovo':1.05,'Asus':0.95,'Samsung':0.90,'Huawei':0.88,'Acer':0.82}
    PW = {'i9':1.55,'Ryzen 9':1.55,'i7':1.32,'Ryzen 7':1.32,'i5':1.12,'Ryzen 5':1.12,'i3':0.90,'Ryzen 3':0.90,'M4':1.45,'M3':1.25,'M2':1.10,'M1':0.95}
    RW = {'64GB':2.05,'32GB':1.55,'24GB':1.32,'16GB':1.12,'8GB':0.95,'4GB':0.82}
    SW = {'NVMe SSD':1.30,'SSD':1.18,'HDD':0.80}
    SZ = {128:0.85,256:0.95,512:1.10,1024:1.30,2048:1.55}
    GW = {'RTX 4070':2.05,'RTX 4060':1.72,'RTX 4050':1.50,'RTX 3050':1.32,'RTX 2050':1.12,'RX 7600':1.22,'Intel Iris Xe':1.02,'Integrated':0.90}
    DW = {17.0:1.20,16.0:1.15,15.6:1.10,14.0:1.00,13.0:0.92}
    REW = {'4K':1.32,'2K':1.20,'FHD':1.10,'HD':0.90}
    RW2 = {240:1.20,165:1.15,144:1.10,120:1.05,60:1.00}
    OW = {'MacOS':1.10,'Windows':1.00,'Linux':0.97,'DOS':0.90}
    TW = {'Yes':1.05,'No':1.00}
    base = 45000.0
    price = []
    for i in range(n):
        mult = W[brands[i%10]]*PW[series[i]]*RW[ram[i]]*SW[st[i]]*SZ[sz[i]]*GW[gpu[i]]*DW[disp[i]]*REW[res[i]]*RW2[ref[i]]*OW[os_[i]]*TW[touch[i]]
        wf = 1.0 - max(0.0, weight[i]-1.8)*0.04
        price.append(base*mult*wf*rng.uniform(0.88,1.12))
    return pd.DataFrame({'Brand':brands,'Processor_Brand':pb,'Processor_Series':series,'RAM':ram,'Storage_Type':st,'Storage_Size':sz,'GPU':gpu,'Display_Size':disp,'Resolution':res,'Refresh_Rate':ref,'Operating_System':os_,'Touch_Screen':touch,'Weight':weight,'Battery':batt,'Price':np.clip(price,25000,600000).round(0)})

if df is None:
    df = generate_synthetic()
print('Training rows:', len(df), '| Price range:', int(df['Price'].min()), '-', int(df['Price'].max()))


## 5. Preprocessing pipeline

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

NUMERIC = ['Storage_Size','Display_Size','Refresh_Rate','Weight','Battery']
CAT = ['Brand','Processor_Brand','Processor_Series','RAM','Storage_Type','GPU','Resolution','Operating_System','Touch_Screen']
TARGET = 'Price'

df = df.dropna(subset=[TARGET]).drop_duplicates().reset_index(drop=True)
for c in NUMERIC + [TARGET]:
    df[c] = pd.to_numeric(df[c], errors='coerce')
for c in CAT:
    df[c] = df[c].astype(str).str.strip().str.title()
# IQR outlier capping
for c in NUMERIC + [TARGET]:
    q1, q3 = df[c].quantile(.25), df[c].quantile(.75); iqr = q3-q1
    df[c] = df[c].clip(q1-1.5*iqr, q3+1.5*iqr)

FEATURE_ORDER = CAT + NUMERIC
pre = ColumnTransformer([
    ('num', Pipeline([('imp',SimpleImputer(strategy='median')),('sc',StandardScaler())]), NUMERIC),
    ('cat', Pipeline([('imp',SimpleImputer(strategy='most_frequent')),('oh',OneHotEncoder(handle_unknown='ignore',sparse_output=False))]), CAT),
])
X = df[FEATURE_ORDER]; y = df[TARGET]
Xp = pre.fit_transform(X)
print('Processed feature matrix:', Xp.shape)


## 6. Train models with hyperparameter tuning

Applies RandomizedSearchCV to tune the top tree-based models. Baseline models (Linear Regression, Decision Tree) are trained without tuning.

In [ ]:
from sklearn.model_selection import train_test_split, cross_val_score, RandomizedSearchCV
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor, GradientBoostingRegressor
from sklearn.ensemble import StackingRegressor
import xgboost as xgb, catboost as cb, lightgbm as lgb

X_train, X_test, y_train, y_test = train_test_split(Xp, y, test_size=0.2, random_state=42)

N_ITER = 15
CV = 3

# --- Baseline models (no tuning) ---
models = {}
models['Linear Regression'] = LinearRegression()
models['Decision Tree'] = DecisionTreeRegressor(random_state=42)

tune_specs = {
  'Random Forest': {
    'model': RandomForestRegressor(random_state=42, n_jobs=-1),
    'dist': {'n_estimators': [200,400,600,800], 'max_depth': [None,12,18,24], 'min_samples_split': [2,5,10], 'min_samples_leaf': [1,2,4]},
  },
  'Extra Trees': {
    'model': ExtraTreesRegressor(random_state=42, n_jobs=-1),
    'dist': {'n_estimators': [200,400,600,800], 'max_depth': [None,12,18,24], 'min_samples_split': [2,5,10]},
  },
  'Gradient Boosting': {
    'model': GradientBoostingRegressor(random_state=42),
    'dist': {'n_estimators': [200,400,600], 'learning_rate': [0.01,0.05,0.1], 'max_depth': [4,6,8], 'subsample': [0.8,1.0]},
  },
  'XGBoost': {
    'model': xgb.XGBRegressor(objective='reg:squarederror', random_state=42, verbosity=0),
    'dist': {'n_estimators': [300,500,800], 'learning_rate': [0.01,0.03,0.05,0.1], 'max_depth': [4,6,8,10], 'subsample': [0.7,0.8,1.0], 'colsample_bytree': [0.7,0.8,1.0], 'reg_alpha': [0,0.1,1.0], 'reg_lambda': [0,0.1,1.0]},
  },
  'CatBoost': {
    'model': cb.CatBoostRegressor(random_seed=42, verbose=False),
    'dist': {'iterations': [300,500,800,1000], 'learning_rate': [0.01,0.03,0.05,0.1], 'depth': [4,6,8,10], 'l2_leaf_reg': [1,3,5,10], 'border_count': [64,128,254]},
  },
  'LightGBM': {
    'model': lgb.LGBMRegressor(random_state=42, verbose=-1),
    'dist': {'n_estimators': [300,500,800,1000], 'learning_rate': [0.01,0.03,0.05,0.1], 'max_depth': [-1,6,8,10], 'num_leaves': [31,64,127,255], 'subsample': [0.7,0.8,1.0], 'colsample_bytree': [0.7,0.8,1.0]},
  },
}

results = {}
importances = {}

# Train baselines
for name in ['Linear Regression', 'Decision Tree']:
    models[name].fit(X_train, y_train)

# Train with tuning
for name, spec in tune_specs.items():
    print(f'Tuning {name}...')
    search = RandomizedSearchCV(spec['model'], spec['dist'], n_iter=N_ITER, cv=CV, scoring='neg_mean_absolute_error', random_state=42, n_jobs=-1)
    search.fit(X_train, y_train)
    models[name] = search.best_estimator_
    print(f'  Best params: {search.best_params_}')

# Evaluate all
for name, mdl in models.items():
    pred = mdl.predict(X_test)
    cv = cross_val_score(mdl, X_train, y_train, cv=5, scoring='neg_mean_absolute_error')
    results[name] = {
        'mae': mean_absolute_error(y_test, pred),
        'mse': mean_squared_error(y_test, pred),
        'rmse': float(np.sqrt(mean_squared_error(y_test, pred))),
        'r2': r2_score(y_test, pred),
        'cv_mae': float(-cv.mean()), 'cv_std': float(cv.std()),
    }
    if hasattr(mdl, 'feature_importances_'):
        importances[name] = mdl.feature_importances_
    print(f'{name:20s} R2={results[name]["r2"]:.4f}  MAE={results[name]["mae"]:.1f}')


## 7. Stacking Ensemble

Combine the top 4 models with a LightGBM meta-learner.

In [ ]:
# Build stacking ensemble from top 4 models
ranked = sorted(results.items(), key=lambda kv:(kv[1]['r2'], -kv[1]['mae']), reverse=True)
top4_names = [t[0] for t in ranked[:4]]
print('Stacking base models:', top4_names)

stack_estimators = [(n.lower().replace(' ','_'), models[n]) for n in top4_names]
stack = StackingRegressor(
    estimators=stack_estimators,
    final_estimator=lgb.LGBMRegressor(n_estimators=300, learning_rate=0.03, max_depth=6, random_state=42, verbose=-1),
    cv=5, n_jobs=-1,
)
stack.fit(X_train, y_train)
pred = stack.predict(X_test)
cv = cross_val_score(stack, X_train, y_train, cv=5, scoring='neg_mean_absolute_error')
results['Stacking Ensemble'] = {
    'mae': mean_absolute_error(y_test, pred),
    'mse': mean_squared_error(y_test, pred),
    'rmse': float(np.sqrt(mean_squared_error(y_test, pred))),
    'r2': r2_score(y_test, pred),
    'cv_mae': float(-cv.mean()), 'cv_std': float(cv.std()),
}
models['Stacking Ensemble'] = stack
print(f'Stacking Ensemble    R2={results["Stacking Ensemble"]["r2"]:.4f}  MAE={results["Stacking Ensemble"]["mae"]:.1f}')


## 8. Select best model & save artifact

In [ ]:
import joblib

best_name = sorted(results.items(), key=lambda kv:(kv[1]['r2'], -kv[1]['mae']), reverse=True)[0][0]
best_model = models[best_name]
print('BEST MODEL:', best_name, 'R2=%.4f' % results[best_name]['r2'])

# Aggregate feature importance
proc_names = list(pre.get_feature_names_out())
fi = {f:0.0 for f in CAT+NUMERIC}
if hasattr(best_model, 'feature_importances_'):
    for nm, imp in zip(proc_names, best_model.feature_importances_):
        orig = nm.split('_',1)[0] if nm.split('_',1)[0] in CAT else nm
        fi[orig] = fi.get(orig,0)+float(imp)
elif best_name == 'Stacking Ensemble' and top4_names:
    # Use the best base model's importances
    base_best = models[top4_names[0]]
    if hasattr(base_best, 'feature_importances_'):
        for nm, imp in zip(proc_names, base_best.feature_importances_):
            orig = nm.split('_',1)[0] if nm.split('_',1)[0] in CAT else nm
            fi[orig] = fi.get(orig,0)+float(imp)
tot = sum(fi.values()) or 1
feature_importance = {k: round(v/tot,4) for k,v in sorted(fi.items(), key=lambda kv: kv[1], reverse=True)}

package = {
    'model': best_model, 'models': models, 'preprocessor': pre,
    'feature_order': FEATURE_ORDER, 'numeric_features': NUMERIC,
    'categorical_features': CAT, 'target_column': TARGET,
    'best_model_name': best_name, 'evaluation_results': results,
    'feature_importance': feature_importance,
    'training_timestamp': pd.Timestamp.now().isoformat(),
    'dataset_info': {'n_samples': int(len(df)), 'price_min': float(y.min()),
                     'price_max': float(y.max()), 'price_mean': float(y.mean())},
}
os.makedirs('/content/LaptopPricePredictor/models', exist_ok=True)
joblib.dump(package, '/content/LaptopPricePredictor/models/best_model.pkl')
with open('/content/LaptopPricePredictor/models/evaluation_metrics.json','w') as f:
    json.dump({'best_model':best_name,'models':results,'feature_importance':feature_importance}, f, indent=2)
print('Saved -> /content/LaptopPricePredictor/models/best_model.pkl')


## 9. Download the artifact

Download `best_model.pkl` and place it in the `models/` folder of the FastAPI backend.

In [ ]:
try:
    from google.colab import files
    files.download('/content/LaptopPricePredictor/models/best_model.pkl')
    files.download('/content/LaptopPricePredictor/models/evaluation_metrics.json')
except Exception as e:
    print('Download skipped (not in Colab):', e)
